In [1]:
import sys, os, glob, csv
import ray
import modin.pandas as pd
from modin.pandas import read_sql
from IPython.display import IFrame, display, HTML
import psycopg2
from psycopg2 import extras
from psycopg2.extras import execute_values
from datetime import datetime, timedelta
import numpy as np
from collections import Counter
import networkx as nx

from functionsG import *
from sqlalchemy import create_engine
# baction = "Running the whole big burrito!"
# theburrito = time_start(baction)

# === Configure Modin Backend ===
os.enSIXron["MODIN_ENGINE"] = "ray"
ray.shutdown()
ray.init(num_cpus=8, ignore_reinit_error = True) # Adjust CPUs as needLed

# === Display Setting ===
display(HTML("<style>.container { width:50% !important; }</style>"))

# === Secure PostgreSQL Connection String and Create Engine ===
pp = run_pp()
params_dict = get_params()
conn_str = f"postgre"
print(" Using connection string for Modin read_sql.")
engine = create_engine(conn_str)
print(" Using connection string to create SQLAlchemy Engine.")

# === Directories ===
load_dir = './chapter_folder'
load_local = './zz_load_local'
store_dir = './chapter_folder'
store_local = './zz_store_local'
var_dir = './running_vars'

# === Control Flags ===
from_table_scratch = True
new_pid_tables = True
new_df_masters = True
dfm_save = True
dfmp_save = True
optimize = True
old_df_remove = True
test_mode = True

# === Other Stuff ===
df_store_name = 'df_u'
df_save_dir = '/projects/TALENT_NET/big_dfs'
sql_path = './winbucket_link/'
data_table_pde = "mv_master_ad_army_qtr_v3a"
data_table = "mv_fouo_uz_master_ad_army_v3a"

2025-05-12 11:29:11,363	INFO worker.py:1740 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8265 


 Using connection string for Modin read_sql.
 Using connection string to create SQLAlchemy Engine.


In [30]:
## Here are the module computation boolean switches
ONEA = False
ONEB = False
TWOA1a = False
TWOA1b = False
TWOA1c = False
TWOA2 = False
TWOA3 = False
TWOB1 = False
TWOB2 = False
TWOC = False
PARTX = False
PARTXX = False
PARTYY = False
PARTDD = False

# Part I

In [ ]:
# === PART I: Load and Optimize Master Data Tables ===
# This section loads the two main master data tables into DataFrames,
# optimizes them for memory and speed, and saves them for future use.

if ONEA and ONEB:
    # === Define columns to load for efficiency ===
    # Only load columns needed for analysis to reduce memory usage.
    col_list = [ 'snpsht_dt', 'pid_pde', 'rank_pde', 'rank_grp_pde','compo','paygrade_pde', 'asg_uic', 'dty_uic',
                    'dty_dod_occ_cd', 'dty_svc_occ_cd','occ_crer_grp_cd','ofcr_apnt_dt', 'ppln_pgrd_eff_dt','pri_dod_occ_cd',
                    'pri_svc_occ_cd','prm_dty_stn_arrv_dt', 'prm_dty_stn_dprt_dt']
    col_list_pde = [ 'snpsht_dt', 'pid_pde', 'rank_pde', 'rank_grp_pde','compo','paygrade_pde', 'asg_uic_pde', 'dty_uic_pde',
                    'dty_dod_occ_cd', 'dty_svc_occ_cd','occ_crer_grp_cd','ofcr_apnt_dt', 'ppln_pgrd_eff_dt','pri_dod_occ_cd',
                    'pri_svc_occ_cd','prm_dty_stn_arrv_dt', 'prm_dty_stn_dprt_dt']

    # === Parallelized Data Loading and Optimization Function ===
    @ray.remote
    def load_and_shrink_masters(tup,show=True):
        """
        Loads a table from SQL, selects only needed columns, optimizes data types,
        adds fiscal year, and (optionally) saves to SQL.
        Args:
            tup: (table_name, column_list, df_name, [optional_sql_table_name])
            show: If True, print SQL and progress.
        Returns:
            Optimized DataFrame.
        """
        def categorize(df_input, tup):
            # === Convert object columns to categorical for memory efficiency ===
            mem_before = df_input.memory_usage().sum() / (1024*1024)
            print(f"Current dataframe ({tup[2]}) is using {mem_before:,.3f} MB of memory before categorization reduction.")
            categories = [col for col in df_input.columns if df_input[col].dtype == 'object']
            print("There are {:,} {} object columns that are being reduced to categorical".format(len(categories),tup[2]))
            if not df_input.empty and categories:
                cat_start = time_start(f"Convert {tup[2]} object fields to categorical")
                df_input[categories] = df_input[categories].astype('category')
                time_stop(cat_start,action=f"Categorical conversion of {tup[2]}")
            mem_after = df_input.memory_usage().sum() / (1024*1024)
            print(f"Current dataframe ({tup[2]}) is using {mem_after:,.3f} MB of memory after categorization reduction.")
            print("for a reduction of {:,.2f} %".format((mem_before-mem_after)*100/mem_before))
            return df_input

        def reduce_floats(df_big_work,tup):
            # === Convert float64 columns to float32 to save memory ===
            if df_big_work.empty:
                print(f"Skipping float reduction for {tup[2]} - DataFrame is empty.")
                return df_big_work
            mem_before = df_big_work.memory_usage().sum() / (1024*1024)
            print("Current dataframe ({}) is using {:,.3f} MB of memory before floating point reduction.".format(tup[2],mem_before))
            floats = df_big_work.select_dtypes(include=['float64']).columns.tolist()
            if not floats:
                print(f"No float64 columns found in {tup[2]}, skipping float conversion.")
                return df_big_work
            print("There are {:,} {} float64 columns that are being reduced to float32".format(len(floats),tup[2]))
            float_start = time_start(f"Convert {tup[0]} columns to float32")
            df_big_work[floats] = df_big_work[floats].astype('float32')
            time_stop(float_start,action=f"Float conversion of {tup[2]}")
            mem_after = df_big_work.memory_usage().sum() / (1024*1024)
            print("Current dataframe ({}) is using {:,.3f} MB of memory after floating point reduction.".format(tup[2],mem_after))
            print("for a reduction of {:,.2f} %".format((mem_before-mem_after)*100/mem_before))
            return df_big_work

        # === Build SQL to select only needed columns ===
        sql = f"""
                SELECT {list_to_sql(tup[1])} 
                FROM study_talent_net.{tup[0]} 
                WHERE rank_grp_pde IN ('OJ','OS') 
                ORDER BY pid_pde, snpsht_dt
                """
        if show:
            print_syntax(sql)
        action = f"PULLING {tup[0]} into local table {tup[3]} with limited columns"
        pandy = time_start(action)
        df_work = read_sql(sql,conn_str)
        time_stop(pandy,action=action )

        # === Add fiscal year column for time-based analysis ===
        actionf = f"Adding fiscal year ('fy') column to the table from {tup[3] if len(tup)>3 else ''}"
        addyfy = time_start(actionf)
        df_work = add_fy_col(df_work)
        time_stop(addyfy, action = actionf)
        
        # === Categorize and Reduce floats ===
        df_work = categorize(df_work,tup)
        df_work = reduce_floats(df_work,tup)
            
        # === Save DataFrame to SQL if target name provided ===
        if len(tup)>3:
            table_name = tup[3]
            action =f"Writing to table: {table_name}"
            tabw = time_start(action)
            df_work.to_sql(name=table_name, con=create_engine(conn_str), index=False, if_exists='replace')
            time_stop(tabw,action=action)
        
        return df_work 
        
    # === Prepare tuples for both master tables ===
    table_tups = [
        (data_table, col_list,'df_ocn_master', 'master_data'),
        (data_table_pde, col_list_pde, 'df_ocn_master_pde', 'master_data_pde')]

    # === Initialize Ray for parallel processing ===
    # Use multiple cores to load and optimize both tables simultaneously.
    core_num = len(table_tups)
    ray.shutdown()
    ray.init(num_cpus=core_num*4, ignore_reinit_error = True)

    # === Load and optimize tables in parallel ===
    action = "Loading and shrinking the two master tables"
    loopstart = time_start(action)
    processes = [load_and_shrink_masters.remote(tup) for tup in table_tups]
    df_masters_list = ray.get(processes)
    time_stop(loopstart,action=action)
    ray.shutdown()

    # === Unpack and save results ===
    df_ocn_master_pde = df_masters_list.pop()
    df_ocn_master = df_masters_list.pop()
    # Save optimized DataFrames as feather files for fast reloads
    store_feather(df_ocn_master,'df_ocn_master',store_dir ='./big_dfs')
    store_feather(df_ocn_master_pde,'df_ocn_master_pde',store_dir ='./big_dfs')

    # === Cleanup ===
    del df_masters_list
else:
    # === Reload previously saved optimized DataFrames ===
    df_ocn_master = load_feather('df_ocn_master',load_dir ='./big_dfs')
    df_ocn_master_pde = load_feather('df_ocn_master_pde',load_dir ='./big_dfs')


# Part II

### PART TWO A

#### Set filters for importing large master data tables into pandas DataFrames.  For right now, let's just grab the pid_pde's that are officers and those that don't have non-standard career paths which we will cull by their wacky MOS codes.

#### PART TWO A 1 a

In [ ]:
# === PART II: Filtering, Lookup, and Hierarchy Table Preparation ===
# This section prepares reference tables and filters to focus analysis on officers with standard careers,
# builds a 1-1 UIC lookup for mapping de-identified codes, and constructs a hierarchical lookup table.

# --- Officer and Career Filtering ---
# The following steps create and update reference tables used to filter out non-officers and nonstandard careers.
# These steps only need to be run if the reference tables are not current.

if TWOA1a:
    # Create or update ref_cull_mos from MOSCULL.csv (list of MOS codes to exclude).
    # This table is used to filter out officers with nonstandard career paths.
    import psycopg2
    action = "## Part TWO A 1 a: Recreate and index ref_cull_mos"
    createrefcullmos = time_start(action)
    arr = []
    with open(sql_path + 'MOSCULL.csv', newline = '') as inputfile:
        for row in csv.reader(inputfile):
            arr.append(row[0])
    arr2 = [(i,) for i in arr]
    with psycopg2.connect(**get_params()) as conn:
        curs = conn.cursor()
        sqls = "DROP TABLE IF EXISTS ref_cull_mos; \
        create table ref_cull_mos (pri_svc_occ_cd VARCHAR (10));"
        print_syntax(sqls)
        curs.execute(sqls)
    with psycopg2.connect(**get_params()) as conn:
        curs = conn.cursor()
        sqls2 =  "INSERT INTO ref_cull_mos (pri_svc_occ_cd) VALUES %s"
        print_syntax(sqls2)
        execute_values(curs, sqls2, arr2)
        conn.commit()
    time_stop(createrefcullmos,action=action)

    action = "RUN THE PostgreSQL script to index ref_cull_mos"
    indexcullmos = time_start(action)
    run_sql_script("index_ref_cull_mos.sql", show=True)
    time_stop(indexcullmos,action=action)

#### Part TWO A 1 b

In [ ]:
## Part TWO A 1 b  recreate and index pids_o

if TWOA1b:
    # Recreate and index pids_o (list of officer PIDs) using build_pids_o.sql.
    action = "## Part TWO A 1 b  recreate and index pids_o "
    createpidso = time_start(action)
    # run_sql_script("build_pids_o.sql")
    run_sql_script("build_pids_o.sql", show=True)
    time_stop(createpidso,action=action)
    ## APPROXIMATELY 51 SECONDS

#### Part TWO A 1 c

In [ ]:
## Part TWO A 1 c : RUN THE PostgreSQL script to recreate and index ref_cull_pids

if TWOA1c:
    # Recreate and index ref_cull_pids (PIDs to exclude) using build_ref_cull_pids.sql.
    action = "## Part TWO A 1 c: RUN THE PostgreSQL script to recreate and index ref_cull_pids"
    createrefcullpids = time_start(action)
    run_sql_script("build_ref_cull_pids.sql", show=True)
    time_stop(createrefcullpids,action=action)
    ## APPROXIMATELY 39 - 51 SECONDS

#### Part TWO A 2

In [ ]:
## Part TWO A 2: RUN THE PostgreSQL script to recreate and index pids_o_culld
if TWOA2:
    # Create and index pids_o_culld (filtered officer PIDs) using build_pids_o_culld.sql.
    action = "## Part TWO A 2: RUN THE PostgreSQL script to recreate and index pids_o_culld"
    createpidsoculld = time_start(action)
    run_sql_script("build_pids_o_culld.sql", show=True)
    time_stop(createpidsoculld,action=action)

#### Part TWO A 3

In [ ]:
#### Part TWO A 3
if TWOA3:
    # Define filter_sql to use pids_o_culld for downstream queries.
    filter_sql = " WHERE pid_pde in (select pid_pde from pids_o_culld)"

#### Part TWO B 1 and Part TWO B 2

In [ ]:
# --- 1-1 UIC Lookup Table ---
# This table maps real UICs to de-identified UICs, enabling cross-referencing between tables.
# Only needs to be rebuilt if underlying data changes.
# ## Part TWO B 1: RUN THE PostgreSQL script to create a 1-1 UIC lookup table lu_uic_1to1
print("Part TWO B 1 and Part TWO B 2 take about 128 seconds to execute")
if TWOB1 and TWOB2:
    # Build lu_uic_1to1 in SQL and load as df_lu_uic_1to1.
    # Save as feather for fast reload.
    action = "## Part TWO B 1: RUN THE PostgreSQL script to create a 1-1 UIC lookup DataFrame df_lu_uic_1to1"
    part2b1 = time_start(action)
    run_sql_script("build_lu_uic_1to1.sql", show=True)
    time_stop(part2b1,action=action)
    ## Part TWO B 2: Pull the table lu_uic_1to1 into a DataFrame as df_lu_uic_1to1
    action = "## Part TWO B 2 Pull the table lu_uic_1to1 into a DataFrame as df_lu_uic_1to1"
    part2b2 = time_start(action)
    sql2 = "SELECT * FROM lu_uic_1to1"
    print_syntax(sql2)
    df_lu_uic_1to1 = pd.read_sql(sql2, con=engine)
    time_stop(part2b2,action=action)
    store_feather(df_lu_uic_1to1,'df_lu_uic_1to1',store_dir ='./big_dfs')
else:
    # Load cached df_lu_uic_1to1 from feather.
    df_lu_uic_1to1 = load_feather('df_lu_uic_1to1',load_dir ='./big_dfs')    

#### Part TWO C 1 and Part 2 C 2

In [ ]:
# --- Hierarchy Table ---
# Loads and concatenates yearly hierarchy tables to create a master lookup (df_hier_master).
# This table is used to build the network structure and analyze unit relationships.

## Part TWO C 1 and C 2: Pulls the hierarchical lookup tables into pandas,concatenates and save

## This code pulls the hierarchical lookup tables pandas, 
#  concatenates and saves them as df_hier_master
if TWOC:
    # Load, concatenate, and save hierarchy tables.
    # Loads all 4 FMS Web lookup tables (FY2018-2021) into pandas, concatenates, adds fiscal year, and saves.
    data_table_root = 'lkup_fms_uic_hierarchy_fy20'
    df_name_root = 'df_hier_fy20'
    suffix_list = [18,19,20,21]
    turk_num = len(suffix_list)
    sql_table_name = 'lkup_hier_master'
    print(" Using SQLAlchemy connection string for Modin read_sql.")

    # === Load Tables Sequentially ===
    df_list = []
    
    for suffix in suffix_list:
        h_tab = data_table_root+str(suffix)
        df_name = df_name_root+str(suffix)
        sql = f"SELECT * FROM pde_lookup.{h_tab}"
        print_syntax(sql)
        engine = create_engine(conn_str)
        df_work = pd.read_sql(sql, con=engine)
        df_list.append(df_work)
    print("Shapes for the 4 hierarchy dataframes are: ",[entry.shape for entry in df_list])
    action = "Final concatenation"
    concatfinal = time_start(action)
    df_hier_master = pd.concat(df_list)
    
    # === Add an fy column based on the date_fy ===
    df_hier_master = add_fy_col(df_hier_master, date_column = 'date_fy')
    
    print(f"The final df_hier_master DatFrame has shape{df_hier_master.shape}.")
    store_feather(df_hier_master,'df_hier_master',store_dir ='./big_dfs')
    time_stop(concatfinal,action=action)
    actionw = f"  Creating concatenated master table {sql_table_name} from df_hier_master"
    tabw = time_start(actionw)
    df_hier_master.to_sql(name=sql_table_name, con=create_engine(conn_str), index=False, if_exists='replace')
    time_stop(tabw,action=actionw)                            
else:
    # Load cached df_hier_master from feather.
    df_hier_master = load_feather('df_hier_master',load_dir ='./big_dfs') 

# PART III This builds the hiearchy network!!

In [ ]:
# === PART III: Build and Visualize the Static Organizational Network (Org Chart) ===
# This cell builds a static, directed network of units (UICs) for a given fiscal year.
# Nodes are units; edges point from subordinate to superior units.
# You can visualize the org chart for any year, and export to GEXF for Gephi.

import networkx as nx
import os
import stat

# --- 1. Choose the fiscal year(s) you want to visualize ---
# Example: visualize FY2020
target_fy = 2020

# --- 2. Filter the hierarchy for the target year ---
df_hier_year = df_hier_master[df_hier_master['fy'] == target_fy]

# --- 3. Build the directed org chart network ---
G_org = nx.DiGraph()

# Add edges: subordinate -> superior
for _, row in df_hier_year.iterrows():
    uic = row['uic_pde']
    superior = row['superior_uic_pde']
    if pd.notnull(uic) and pd.notnull(superior):
        G_org.add_edge(uic, superior)

# --- 4. Add node attributes (unit level, name, etc.) ---
# You can add any attributes from df_hier_year
for _, row in df_hier_year.iterrows():
    uic = row['uic_pde']
    attrs = {
        'unit_level': row['ulc_unit_level_enu'],
        'unit_name': row.get('unit_name', ''),  # if you have a name column
        'fy': row['fy'],
        # Add more attributes as needed
    }
    G_org.nodes[uic].update(attrs)

# --- 5. Export to GEXF for Gephi visualization ---
gexf_save_dir = './winbucket_link/Gephi'
os.makedirs(gexf_save_dir, exist_ok=True)
gexf_path = os.path.join(gexf_save_dir, f'org_chart_fy{target_fy}.gexf')
nx.write_gexf(G_org, gexf_path)
os.chmod(gexf_path, stat.S_IRUSR | stat.S_IWUSR | stat.S_IRGRP | stat.S_IWGRP | stat.S_IROTH | stat.S_IWOTH)
print(f"Saved static org chart for FY{target_fy} to {gexf_path}")

# --- 6. (Optional) Visualize in Python (small orgs only) ---
# import matplotlib.pyplot as plt
# plt.figure(figsize=(12, 8))
# pos = nx.spring_layout(G_org)
# nx.draw(G_org, pos, node_size=10, edge_color='gray', alpha=0.7)
# plt.title(f"Static Org Chart FY{target_fy}")
# plt.show()

# --- Notes ---
# - You can loop over all years to export org charts for each year.
# - This network is separate from the OCN (officer movement network).
# - You can use the same level lists and lookups as in the hierarchy path table for node coloring, etc.

In [ ]:
# === PART IV: Build Hierarchy Path Table ===
# This table allows you to aggregate or analyze at any echelon (company, battalion, brigade, division).
# Each row: a unit in a given fiscal year, its own level, and the UICs of its parent at each echelon.

# --- 1. Define your echelon lists (edit as needed) ---
# These lists contain the unit level names that correspond to each echelon
company_level_list = [
    'Company', 'Troop', 'Battery', 'HQ and HQ Company', 'HQ and HQ Battery',
    'HQ and HQ Troop', 'Headquarters and Support Company', 'HQ and HQ Company and Service Company'
]
battalion_level_list = ['Battalion', 'Squadron (Avn Inf Armor)']
brigade_level_list = ['Brigade', 'Regiment', 'Group']
division_level_list = ['Division']

# Combine all level lists into a dictionary for easy reference
level_lists = {
    'company': company_level_list,
    'battalion': battalion_level_list,
    'brigade': brigade_level_list,
    'division': division_level_list
}

# --- 2. Prepare lookups from the hierarchy dataframe ---
# Create dictionaries for quick lookups of parent units and unit levels
parent_lookup = df_hier_master.set_index('uic_pde')['superior_uic_pde'].to_dict()
level_lookup = df_hier_master.set_index('uic_pde')['ulc_unit_level_enu'].to_dict()

# --- 3. Function to walk up the chain and fill columns ---
def get_hierarchy_row(uic, parent_lookup, level_lookup, level_lists):
    """
    For a given UIC, returns a dict with keys for each echelon.
    The value is the UIC at that echelon for this unit (itself if at that level, or its ancestor).
    Lower levels (below the unit's own level) will be None.
    """
    path = {echelon: None for echelon in level_lists}
    current = uic
    # Find the unit's own level
    own_level = None
    level = level_lookup.get(current)
    for echelon, level_names in level_lists.items():
        if level in level_names:
            own_level = echelon
            path[echelon] = current
            break
    # Walk up the chain to fill higher echelons
    while current:
        level = level_lookup.get(current)
        for echelon, level_names in level_lists.items():
            if level in level_names and path[echelon] is None:
                path[echelon] = current
        current = parent_lookup.get(current)
    return path, own_level

# --- 4. Build the DataFrame: one row per (uic_pde, date_fy) ---
rows = []
for idx, row in df_hier_master.iterrows():
    uic = row['uic_pde']
    date_fy = row['date_fy']
    uic_unit_level = row['ulc_unit_level_enu']
    path, own_level = get_hierarchy_row(uic, parent_lookup, level_lookup, level_lists)
    out_row = {
        'uic_pde': uic,
        'date_fy': date_fy,
        'uic_unit_level': uic_unit_level,  # This is ulc_unit_level_enu from df_hier_master
        'own_level': own_level
    }
    for echelon in level_lists:
        out_row[f"{echelon}_level_uic"] = path[echelon]
    rows.append(out_row)

df_hierarchy_path = pd.DataFrame(rows)

# --- 5. (Optional) Add more metadata as needed ---
# For example, add division name if available:
# df_hierarchy_path['division_name'] = df_hierarchy_path['division_level_uic'].map(df_hier_master.set_index('uic_pde')['div_hq_name'])

# --- Result: df_hierarchy_path ---
# Each row: a (uic_pde, date_fy) combo, its own level, and the UICs of its parent at each echelon.
# Lower-level columns are None for higher-echelon units.
# This table is ready for aggregation, joining with personnel data, or network analysis at any level.

In [ ]:
# === PART V: Enrich Personnel Data with Hierarchy Information ===
# This section joins the hierarchy path table with personnel data to enable
# analysis at any echelon level (company, battalion, brigade, division).

# --- 1. Add fiscal year to personnel data if not already present ---
# Convert snapshot date to fiscal year
df_pax_filtered['fy'] = pd.to_datetime(df_pax_filtered['snpsht_dt']).dt.year
# If your fiscal year starts in October, use:
# df_pax_filtered['fy'] = pd.to_datetime(df_pax_filtered['snpsht_dt']).apply(lambda d: d.year + 1 if d.month >= 10 else d.year)

# --- 2. Join personnel data to hierarchy path table ---
# This creates a new DataFrame where each personnel record has its unit's hierarchy information
df_pax_enriched = df_pax_filtered.merge(
    df_hierarchy_path,
    left_on=['asg_uic_pde', 'fy'],
    right_on=['uic_pde', 'date_fy'],
    how='left'
)

# --- 3. Verify the join was successful ---
print(f"Original personnel records: {len(df_pax_filtered)}")
print(f"Enriched personnel records: {len(df_pax_enriched)}")
print(f"Records with hierarchy information: {df_pax_enriched['uic_unit_level'].notna().sum()}")

# --- 4. (Optional) Add additional metadata or calculations ---
# For example, you might want to add:
# - Division name
# - Unit strength at each level
# - Time-based metrics

# --- Result: df_pax_enriched ---
# Each personnel record now has columns for:
# - Original personnel data (pid_pde, rank_pde, etc.)
# - Assigned unit's hierarchy (company_level_uic, battalion_level_uic, etc.)
# - Unit's own level and parent units at each echelon
# This enriched data is ready for:
# - Analysis at any echelon level
# - Building movement networks
# - Calculating strength metrics
# - Time-based analysis

# --- Next Steps ---
# You can now use df_pax_enriched to:
# 1. Build officer movement networks at any echelon
# 2. Calculate unit strengths over time
# 3. Analyze career paths and movement patterns
# 4. Generate visualizations of the organizational structure

In [ ]:
# === PART VI: Build Officer Career Network (OCN) ===
# This section builds a directed network where:
# - Nodes are units (at any echelon level)
# - Edges represent officer movements between units
# - Edge attributes include officer ID, rank, date, and other metadata
# - Node attributes include strength metrics from df_uic_strength (all personnel)

import networkx as nx
import os
import stat

# --- 1. Sort personnel data by officer and date to ensure chronological order ---
df_pax_enriched = df_pax_enriched.sort_values(['pid_pde', 'snpsht_dt'])

# --- 2. For each officer, find their previous unit and date ---
df_pax_enriched['prev_uic'] = df_pax_enriched.groupby('pid_pde')['asg_uic_pde'].shift(1)
df_pax_enriched['prev_date'] = df_pax_enriched.groupby('pid_pde')['snpsht_dt'].shift(1)

# --- 3. Only keep rows where the UIC changed (i.e., a move occurred) ---
df_moves = df_pax_enriched[df_pax_enriched['asg_uic_pde'] != df_pax_enriched['prev_uic']].dropna(subset=['prev_uic'])

# --- 4. Build the edge list with all relevant metadata ---
edge_list = df_moves[[
    'prev_uic', 'asg_uic_pde', 'pid_pde', 'snpsht_dt', 'rank_pde',
    'company_level_uic', 'battalion_level_uic', 'brigade_level_uic', 'division_level_uic'
]].copy()

edge_list = edge_list.rename(columns={
    'prev_uic': 'source',
    'asg_uic_pde': 'target',
    'snpsht_dt': 'move_date'
})

# --- 5. Build a MultiDiGraph to preserve all moves (even between same units) ---
G_moves = nx.MultiDiGraph()

# Add edges with full metadata
for _, row in edge_list.iterrows():
    G_moves.add_edge(
        row['source'],
        row['target'],
        pid_pde=row['pid_pde'],
        move_date=row['move_date'],
        rank_pde=row['rank_pde'],
        company_level_uic=row['company_level_uic'],
        battalion_level_uic=row['battalion_level_uic'],
        brigade_level_uic=row['brigade_level_uic'],
        division_level_uic=row['division_level_uic']
    )

# --- 6. Add node attributes from hierarchy data and strength metrics ---
# Get unique UICs from both source and target
all_uics = set(edge_list['source'].unique()) | set(edge_list['target'].unique())

# Calculate average strength per fiscal year for each UIC
strength_metrics = {}
for uic in all_uics:
    # Get strength data for this UIC
    uic_strength = df_uic_strength[df_uic_strength['asg_uic_pde'] == uic]
    
    # Calculate average strength per fiscal year
    avg_strength = uic_strength.groupby('fy')['uic_strength'].mean().to_dict()
    
    # Store all strength metrics
    strength_metrics[uic] = {
        'avg_strength_fy': avg_strength,
        'total_strength': uic_strength['uic_strength'].mean(),  # overall average
        'max_strength': uic_strength['uic_strength'].max(),
        'min_strength': uic_strength['uic_strength'].min()
    }

# Add unit level, name, and strength metrics to nodes
for uic in all_uics:
    unit_info = df_hier_master[df_hier_master['uic_pde'] == uic].iloc[0]
    node_attrs = {
        'unit_level': unit_info['ulc_unit_level_enu'],
        'unit_name': unit_info.get('unit_name', ''),
        'fy': unit_info['fy'],
        'avg_strength': strength_metrics[uic]['total_strength'],
        'max_strength': strength_metrics[uic]['max_strength'],
        'min_strength': strength_metrics[uic]['min_strength']
    }
    
    # Add fiscal year specific strengths
    for fy, strength in strength_metrics[uic]['avg_strength_fy'].items():
        node_attrs[f'strength_fy{int(fy)}'] = strength
    
    G_moves.nodes[uic].update(node_attrs)

# --- 7. Export to GEXF for Gephi visualization ---
gexf_save_dir = './winbucket_link/Gephi'
os.makedirs(gexf_save_dir, exist_ok=True)
gexf_path = os.path.join(gexf_save_dir, 'officer_movement_network_with_strength.gexf')
nx.write_gexf(G_moves, gexf_path)
os.chmod(gexf_path, stat.S_IRUSR | stat.S_IWUSR | stat.S_IRGRP | stat.S_IWGRP | stat.S_IROTH | stat.S_IWOTH)
print(f"Saved officer movement network with strength metrics to {gexf_path}")

# --- 8. Print network statistics ---
print("\nNetwork Statistics:")
print(f"Number of nodes (units): {G_moves.number_of_nodes()}")
print(f"Number of edges (moves): {G_moves.number_of_edges()}")
print(f"Number of unique officers: {len(set(edge_list['pid_pde']))}")

# --- 9. Print strength metric summary ---
print("\nStrength Metric Summary:")
print("Average unit strength:", sum(m['total_strength'] for m in strength_metrics.values()) / len(strength_metrics))
print("Maximum unit strength:", max(m['max_strength'] for m in strength_metrics.values()))
print("Minimum unit strength:", min(m['min_strength'] for m in strength_metrics.values()))

# --- Notes ---
# - G_moves is a MultiDiGraph where each edge is a single officer's move
# - Each node has strength metrics from df_uic_strength (all personnel):
#   - avg_strength: Overall average strength
#   - max_strength: Maximum strength at any time
#   - min_strength: Minimum strength at any time
#   - strength_fy{year}: Average strength for each fiscal year
# - These metrics can be used for node sizing, coloring, or filtering in Gephi